# FTMO AI Trading System — Cloud Inference
**Runs LightGBM + CatBoost inference on all 8 pretrained symbol pairs.**

### Open this notebook directly from GitHub (no Google Drive needed):
```
https://colab.research.google.com/github/Codebreaker-source/AI-Trading-System/blob/main/src/ftmo_system/colab/trading_inference.ipynb
```
Models and code are pulled from GitHub automatically — updates deploy on next session start.

### One-time setup:
1. Go to **Colab Secrets** (🔑 icon in left sidebar)
2. Add secret named `AZURE_STORAGE_CONNECTION_STRING` — paste your full Azure connection string
3. Toggle **Notebook access** ON for that secret
4. **Runtime → Run all**

The inference loop in the last cell runs forever. Leave this tab open during trading hours.

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install -q azure-storage-blob lightgbm catboost xgboost joblib pandas numpy

In [ ]:
# ── Cell 2: Clone repo to get models + feature_expander ─────────────────
import os
if not os.path.exists('/content/AI-Trading-System'):
    !git clone -q https://github.com/Codebreaker-source/AI-Trading-System.git /content/AI-Trading-System
else:
    !git -C /content/AI-Trading-System pull -q

import sys
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system')
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system/core')
print('Repo ready.')
!ls /content/AI-Trading-System/models/current/

In [ ]:
# ── Cell 3: Load Azure connection string from Colab Secrets ─────────────
from google.colab import userdata
import os

try:
    AZURE_CONN_STR = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
    os.environ['AZURE_STORAGE_CONNECTION_STRING'] = AZURE_CONN_STR
    print('Azure connection string loaded from Colab Secrets.')
except Exception as e:
    raise RuntimeError(
        'AZURE_STORAGE_CONNECTION_STRING not found in Colab Secrets. '
        'Add it via the key icon in the left sidebar.'
    ) from e

In [ ]:
# ── Cell 4: Load ALL available models dynamically ───────────────────────
# Scans models/current/ for:
#   *.joblib          → LightGBM / CatBoost / XGBoost  (sklearn-style)
#   transformer/*.pth → SimpleTransformer               (PyTorch)
# No hardcoded symbol list — new models are picked up automatically.

import joblib, glob, os
import torch
import torch.nn as nn

MODEL_DIR = '/content/AI-Trading-System/models/current'

# ── PyTorch model architectures ──────────────────────────────────────────

class SimpleTransformer(nn.Module):
    """Tabular Transformer: 105 flat features → 3 classes (0=SELL 1=HOLD 2=BUY)"""
    def __init__(self, input_dim=105, d_model=64, nhead=4, num_layers=2,
                 num_classes=3, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier  = nn.Linear(d_model, num_classes)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        x = x.unsqueeze(1)           # [batch, 1, 105]
        x = self.embedding(x)        # [batch, 1, d_model]
        x = self.transformer(x)      # [batch, 1, d_model]
        x = x.squeeze(1)             # [batch, d_model]
        x = self.dropout(x)
        return self.classifier(x)    # [batch, 3]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')

# ── Load sklearn models (.joblib) ────────────────────────────────────────
lgbm_models     = {}
catboost_models = {}

for path in sorted(glob.glob(f'{MODEL_DIR}/*.joblib')):
    fname = os.path.basename(path)
    if fname.endswith('_lightgbm.joblib'):
        base = fname[:-len('_lightgbm.joblib')]
        lgbm_models[base] = joblib.load(path)
        print(f'  LGB loaded: {base}')
    elif fname.endswith('_catboost.joblib'):
        base = fname[:-len('_catboost.joblib')]
        catboost_models[base] = joblib.load(path)
        print(f'  CAT loaded: {base}')

# ── Load Transformer models (.pth) ───────────────────────────────────────
transformer_models = {}

for path in sorted(glob.glob(f'{MODEL_DIR}/transformer/*.pth')):
    fname = os.path.basename(path)                      # EURUSD.sim_transformer.pth
    base  = fname.replace('_transformer.pth', '')       # EURUSD.sim  →  strip .sim below
    # Normalise to base without broker suffix
    base_clean = base.replace('.sim', '').replace('.i', '').upper()

    model = SimpleTransformer(input_dim=105).to(DEVICE)
    state = torch.load(path, map_location=DEVICE)
    # .pth may contain full checkpoint dict or bare state_dict
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state)
    model.eval()
    transformer_models[base_clean] = model
    print(f'  TRF loaded: {base_clean}')

print(f'\nLoaded: {len(lgbm_models)} LGB  {len(catboost_models)} CAT  {len(transformer_models)} Transformer')

In [ ]:
# ── Cell 5: Load feature expander ───────────────────────────────────────
from feature_expander import expand_features
import numpy as np
import pandas as pd
print('Feature expander loaded.')

In [ ]:
# ── Cell 6: Azure helpers ────────────────────────────────────────────────
from azure.storage.blob import BlobServiceClient
import json
from datetime import datetime, timezone
import io

FEATURES_CONTAINER    = 'trading-features'
PREDICTIONS_CONTAINER = 'colab-predictions'

blob_service = BlobServiceClient.from_connection_string(AZURE_CONN_STR)

# Ensure containers exist
for c in [FEATURES_CONTAINER, PREDICTIONS_CONTAINER]:
    try:
        blob_service.create_container(c)
        print(f'Created container: {c}')
    except:
        print(f'Container exists: {c}')

def download_features(symbol: str):
    """Download feature CSV for a symbol from Azure. Returns DataFrame or None."""
    try:
        blob = blob_service.get_blob_client(container=FEATURES_CONTAINER,
                                            blob=f'{symbol}_features.csv')
        raw = blob.download_blob().readall().decode('utf-8')
        df  = pd.read_csv(io.StringIO(raw))
        return df
    except Exception:
        return None

def upload_prediction(symbol: str, source: str, action: str, confidence: float):
    """Upload inference result to Azure."""
    payload = {
        'symbol':     symbol,
        'source':     source,
        'action':     action,
        'confidence': round(float(confidence), 4),
        'timestamp':  datetime.now(timezone.utc).isoformat(),
    }
    blob = blob_service.get_blob_client(container=PREDICTIONS_CONTAINER,
                                        blob=f'{symbol}_{source}_pred.json')
    blob.upload_blob(json.dumps(payload), overwrite=True)

print('Azure helpers ready.')

In [ ]:
# ── Cell 7: Prediction helpers (HOLD-bias fix, all model types) ──────────
#
# Label encoding: 0=SELL  1=HOLD  2=BUY
# HOLD-bias fix: if HOLD wins but a directional class ≥ BIAS_THRESHOLD, use it.

import numpy as np

BIAS_THRESHOLD = 0.35   # directional prob must exceed this to override HOLD
LABEL_MAP      = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}


def predict_sklearn(model, features_105: np.ndarray):
    """Run sklearn-style predict_proba() with HOLD-bias fix. Returns (action, conf)."""
    try:
        proba    = model.predict_proba(features_105.reshape(1, -1))[0]
        sell_p   = float(proba[0])
        hold_p   = float(proba[1])
        buy_p    = float(proba[2])
        pred_idx = int(np.argmax(proba))

        if pred_idx == 1:   # HOLD wins — apply bias fix
            if buy_p >= BIAS_THRESHOLD and buy_p > sell_p:
                return 'BUY',  buy_p
            if sell_p >= BIAS_THRESHOLD and sell_p > buy_p:
                return 'SELL', sell_p
            return 'HOLD', hold_p

        return LABEL_MAP[pred_idx], float(proba[pred_idx])
    except Exception as e:
        print(f'sklearn predict error: {e}')
        return 'HOLD', 0.0


def predict_transformer(model, features_105: np.ndarray):
    """Run PyTorch SimpleTransformer inference with HOLD-bias fix. Returns (action, conf)."""
    try:
        x     = torch.tensor(features_105, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)                     # [1, 3]
            proba  = torch.softmax(logits, dim=1).cpu().numpy()[0]

        sell_p   = float(proba[0])
        hold_p   = float(proba[1])
        buy_p    = float(proba[2])
        pred_idx = int(np.argmax(proba))

        if pred_idx == 1:   # HOLD wins — apply bias fix
            if buy_p >= BIAS_THRESHOLD and buy_p > sell_p:
                return 'BUY',  buy_p
            if sell_p >= BIAS_THRESHOLD and sell_p > buy_p:
                return 'SELL', sell_p
            return 'HOLD', hold_p

        return LABEL_MAP[pred_idx], float(proba[pred_idx])
    except Exception as e:
        print(f'transformer predict error: {e}')
        return 'HOLD', 0.0


def run_inference_for_symbol(symbol_full: str, base: str):
    """
    Download features, expand 27→105, run LGB + CatBoost + Transformer,
    upload non-HOLD predictions to Azure.

    Returns: (lgbm_result, catboost_result, transformer_result)
    Each result is (action: str, confidence: float) or None if no model/data.
    """
    df = download_features(symbol_full)
    if df is None or df.empty:
        return None, None, None

    latest = df.iloc[-1].values.astype(np.float32)
    if len(latest) < 27:
        return None, None, None
    features_27 = latest[:27]

    try:
        features_105 = expand_features(features_27)
    except Exception as e:
        print(f'Feature expand failed {base}: {e}')
        return None, None, None

    lgbm_result = catboost_result = transformer_result = None

    # ── LightGBM ──────────────────────────────────────────────────────────
    if base in lgbm_models:
        action, conf = predict_sklearn(lgbm_models[base], features_105)
        lgbm_result  = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'lgbm', action, conf)

    # ── CatBoost ──────────────────────────────────────────────────────────
    if base in catboost_models:
        action, conf    = predict_sklearn(catboost_models[base], features_105)
        catboost_result = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'catboost', action, conf)

    # ── Transformer ───────────────────────────────────────────────────────
    if base in transformer_models:
        action, conf      = predict_transformer(transformer_models[base], features_105)
        transformer_result = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'transformer', action, conf)

    return lgbm_result, catboost_result, transformer_result


print('Inference helpers ready (LGB + CatBoost + Transformer).')

In [ ]:
# ── Cell 8: Dynamic symbol discovery ────────────────────────────────────
# Instead of a hardcoded 8-symbol list, we:
#   1. Scan models/current/ for all *_lightgbm.joblib and *_catboost.joblib files
#   2. Scan Azure trading-features container for symbols with live feature CSVs
#   3. Build SYMBOL_MAP from the intersection — any symbol that has BOTH a model
#      AND fresh features from the EA gets inference run on it automatically.
#
# This means as daily_retrainer.py creates new models and they get pushed to
# GitHub, Colab picks them up on the next session start with no code changes.

import glob
import os

MODEL_DIR = '/content/AI-Trading-System/models/current'

def discover_symbol_map() -> dict:
    """
    Returns { 'EURUSD.sim': 'EURUSD', 'GBPUSD.sim': 'GBPUSD', ... }
    for every base symbol that has at least one model (lgbm or catboost)
    AND has a live feature CSV in Azure.
    """
    # Step 1: find all base symbols with trained models
    model_bases = set()
    for path in glob.glob(f'{MODEL_DIR}/*.joblib'):
        fname = os.path.basename(path)
        for suffix in ('_lightgbm.joblib', '_catboost.joblib', '_xgboost.joblib'):
            if fname.endswith(suffix):
                base = fname[:-len(suffix)]
                model_bases.add(base)

    print(f'Model bases found: {sorted(model_bases)}')

    # Step 2: find all symbols with live feature CSVs in Azure
    live_symbols = set()
    try:
        container = blob_service.get_container_client(FEATURES_CONTAINER)
        for blob in container.list_blobs():
            name = blob.name  # e.g. 'EURUSD.sim_features.csv'
            if name.endswith('_features.csv'):
                sym = name[:-len('_features.csv')]  # e.g. 'EURUSD.sim'
                live_symbols.add(sym)
        print(f'Live feature symbols in Azure: {sorted(live_symbols)}')
    except Exception as e:
        print(f'Could not list Azure features container: {e}')
        # Fallback: build from model bases assuming .sim suffix
        live_symbols = {f'{b}.sim' for b in model_bases}

    # Step 3: build map — strip .sim suffix to match model base names
    symbol_map = {}
    for sym_full in sorted(live_symbols):
        # Strip known broker suffixes to get base name
        base = sym_full
        for sfx in ('.sim', '.i', '_SB', '.r', '.a', '.b', '.m', '.pro'):
            if base.endswith(sfx):
                base = base[:-len(sfx)]
                break
        base = base.upper()
        if base in model_bases:
            symbol_map[sym_full] = base

    return symbol_map


SYMBOL_MAP = discover_symbol_map()
print(f'\nActive inference symbols ({len(SYMBOL_MAP)}):')
for full, base in sorted(SYMBOL_MAP.items()):
    lgbm_ok = base in lgbm_models
    cb_ok   = base in catboost_models
    print(f'  {full:<20} LGB={lgbm_ok}  CAT={cb_ok}')

In [ ]:
# ── Cell 9: MAIN INFERENCE LOOP — runs forever ───────────────────────────
# Leave this cell running. It polls Azure every 30 seconds,
# runs LGB + CatBoost + Transformer inference on all available symbols,
# and uploads non-HOLD predictions so live_trading_system.py can consume them.

import time
from IPython.display import clear_output

POLL_INTERVAL = 30   # seconds between inference runs
cycle = 0

n_lgb = len(lgbm_models)
n_cat = len(catboost_models)
n_trf = len(transformer_models)

print('=== FTMO AI Trading System — Colab Inference ===')
print(f'Models: {n_lgb} LGB  +  {n_cat} CatBoost  +  {n_trf} Transformer')
print(f'Symbols: {list(SYMBOL_MAP.keys())}')
print(f'Poll interval: {POLL_INTERVAL}s')
print('Running... (interrupt kernel to stop)')
print()

while True:
    cycle += 1
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    results_log = []

    for sym_full, base in SYMBOL_MAP.items():
        lgbm_r, cb_r, trf_r = run_inference_for_symbol(sym_full, base)

        def fmt(r):
            if r is None:  return '—'
            return f'{r[0]} ({r[1]:.2f})'

        results_log.append(
            f'  {base:<8}  LGB: {fmt(lgbm_r):<20}  CAT: {fmt(cb_r):<20}  TRF: {fmt(trf_r)}'
        )

    clear_output(wait=True)
    print(f'=== Cycle {cycle} · {ts} ===')
    for line in results_log:
        print(line)
    print(f'\nNext run in {POLL_INTERVAL}s...')

    time.sleep(POLL_INTERVAL)